In [1]:
# Install necessary Hugging Face libraries
!pip install -q transformers datasets accelerate evaluate

In [2]:
import os
import pandas as pd
import numpy as np

# Configure Kaggle API credentials (only works if kaggle.json is uploaded to /content/)
if os.path.exists('kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d snap/amazon-fine-food-reviews -f Reviews.csv --unzip
    df = pd.read_csv('Reviews.csv')
else:
    print("Kaggle token not found. We will generate/simulate a mock sample to ensure the code executes safely without errors.")
    # Generate dummy data mimicking the Amazon Reviews dataset
    np.random.seed(42)
    dummy_data = {
        'Text': [
            "This food is amazing! My dog absolutely loves it.",
            "Okay quality, but way too expensive for what it is.",
            "Horrible product. Arrived broken and tasted stale.",
            "Great taste, fast shipping, and good packaging.",
            "It is alright, nothing special. Neutral feelings.",
            "Disgusted by the smell. Do not buy!"
        ] * 150,
        'Score': [5, 3, 1, 5, 3, 1] * 150
    }
    df = pd.DataFrame(dummy_data)

# Inspect the data structure
display(df.head())

Kaggle token not found. We will generate/simulate a mock sample to ensure the code executes safely without errors.


,Text,Score
0,This food is amazing! My dog absolutely loves it.,5
1,"Okay quality, but way too expensive for what i...",3
2,Horrible product. Arrived broken and tasted st...,1
3,"Great taste, fast shipping, and good packaging.",5
4,"It is alright, nothing special. Neutral feelings.",3


### Step 3: Data Preprocessing
We map the review `Score` (1 to 5 stars) to 3 sentiment categories:
* **Negative (0)**: Stars 1 & 2
* **Neutral (1)**: Star 3
* **Positive (2)**: Stars 4 & 5

In [3]:
from sklearn.model_selection import train_test_split

def map_sentiment(score):
    if score <= 2:
        return 0 # Negative
    elif score == 3:
        return 1 # Neutral
    else:
        return 2 # Positive

# Map scores and keep a subset for faster training demonstration
df['label'] = df['Score'].apply(map_sentiment)
df_subset = df[['Text', 'label']].rename(columns={'Text': 'text'}).sample(min(2000, len(df)), random_state=42).reset_index(drop=True)

# Split into train and test sets
train_df, test_df = train_test_split(df_subset, test_size=0.2, random_state=42, stratify=df_subset['label'])
print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
print(train_df['label'].value_counts(normalize=True))

Train size: 720 | Test size: 180
label
2    0.333333
1    0.333333
0    0.333333
Name: proportion, dtype: float64


### Step 4: LSTM Baseline Model
We tokenize words, convert them to integer sequences, pad them, and build an LSTM network using Keras.

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# Parameters
MAX_WORDS = 10000
MAX_LEN = 100
EMBEDDING_DIM = 64

# Tokenization
tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(train_df['text'])

X_train_seq = pad_sequences(tok.texts_to_sequences(train_df['text']), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq = pad_sequences(tok.texts_to_sequences(test_df['text']), maxlen=MAX_LEN, padding='post', truncating='post')

y_train = tf.keras.utils.to_categorical(train_df['label'], num_classes=3)
y_test = tf.keras.utils.to_categorical(test_df['label'], num_classes=3)

# Build LSTM
lstm_model = Sequential([
    Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

# Train LSTM
history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.7762 - loss: 1.0073 - val_accuracy: 1.0000 - val_loss: 0.7893
Epoch 2/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 1.0000 - loss: 0.4113 - val_accuracy: 1.0000 - val_loss: 0.0824
Epoch 3/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 1.0000 - loss: 0.0418 - val_accuracy: 1.0000 - val_loss: 0.0094
Epoch 4/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 1.0000 - loss: 0.0102 - val_accuracy: 1.0000 - val_loss: 0.0039
Epoch 5/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 1.0000 - loss: 0.0061 - val_accuracy: 1.0000 - val_loss: 0.0024


### Step 5: Fine-Tuning BERT (Hugging Face Transformer)
We use Hugging Face `datasets` and `transformers` to fine-tune `distilbert-base-uncased` for classification.

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Convert DataFrames to Hugging Face Dataset
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

# Load pre-trained Tokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_tokenized = train_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

# Load pre-trained BERT Sequence Classifier
bert_model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)

# Setup metric
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Trainer object
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

# Fine-tune BERT
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Step 6: Evaluation and Comparison
Now we evaluate both models on the test split and discuss transfer learning efficiency and real-world deployment challenges.

In [3]:
# Ensure all required variables are in memory before evaluating
try:
    # Evaluate LSTM baseline
    lstm_eval = lstm_model.evaluate(X_test_seq, y_test, verbose=0)
    print(f"LSTM Baseline Accuracy: {lstm_eval[1]:.4f}")

    # Evaluate BERT model
    bert_eval = trainer.evaluate()
    print(f"BERT fine-tuned Accuracy: {bert_eval['eval_accuracy']:.4f}")
except NameError as e:
    print(f"Error: {e}")
    print("\nPlease make sure you run the previous cells in order (Step 3, Step 4, and Step 5) to train and load the models into the runtime memory first.")

Error: name 'lstm_model' is not defined

Please make sure you run the previous cells in order (Step 3, Step 4, and Step 5) to train and load the models into the runtime memory first.


## Summary of Trade-offs & Discussion

| Feature | LSTM Baseline | Fine-Tuned BERT |
| :--- | :--- | :--- |
| **Accuracy / F1** | Moderate (relies heavily on dataset size) | High (leverages pre-trained context semantics) |
| **Training Speed** | Fast (on CPU/GPU, light footprint) | Slow (demands high-end GPUs like T4) |
| **Parameters** | Small (~1M parameters) | Large (~66M for DistilBERT) |
| **Cold Start / Data**| Needs a lot of domain text to build embeddings | Excellent for zero-shot or few-shot tuning |

### Real-world Deployment Constraints
1. **Latency & Cost**: Large models like BERT require expensive GPU runtimes in production. In contrast, smaller sequence models (like LSTM or DistilBERT optimized with ONNX runtime) can be deployed cost-effectively on CPUs or edge devices.
2. **Drift & Maintenance**: Text terminology changes over time. Pre-trained representations might slowly lose alignment with newer colloquialisms, demanding periodic parameter tuning.